# **Install Dependencies**

In [1]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


# **Import Required Libraries** 

In [2]:
import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

# **Load the 3D Protein Structure**

In [3]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("TP53", "/kaggle/input/datasets/aratipatil25/protein-sequence/1TSR.pdb")

# **Remove Water Molecules**

In [4]:
for model in structure:
    for chain in model:
        for res in list(chain):
            if res.id[0] == 'W':
                chain.detach_child(res.id)

# **Compute Solvent Accessible Surface Area (SASA)**

In [5]:
sr = ShrakeRupley(probe_radius=1.4)
sr.compute(structure, level="R")

# **Define Chains and Target Mutations**

In [6]:
chain_B = structure[0]['B'] # The TP53 Protein
# Isolate DNA atoms (Chains E and F) to calculate how close mutations are to the DNA
dna_atoms = [atom for chain in ['E', 'F'] for res in structure[0][chain] for atom in res.get_atoms()]

# Our target wild-type amino acids and their positions
# V143, R175, Y220, R248, R273
mutations = [
    {'pos': 143, 'aa': 'VAL', 'max_sasa': 157.0},
    {'pos': 175, 'aa': 'ARG', 'max_sasa': 241.0},
    {'pos': 220, 'aa': 'TYR', 'max_sasa': 229.0},
    {'pos': 248, 'aa': 'ARG', 'max_sasa': 241.0},
    {'pos': 273, 'aa': 'ARG', 'max_sasa': 241.0}
]

print("--- Structural Analysis Results ---\n")

--- Structural Analysis Results ---



# **Calculate and Classify Relative SASA**

In [7]:
for mut in mutations:
    res_seq = mut['pos']
    res = chain_B[res_seq]
    
    # Calculate Relative SASA (%)
    # Absolute SASA (Å²) divided by the theoretical maximum SASA for that specific amino acid
    absolute_sasa = res.sasa
    relative_sasa = (absolute_sasa / mut['max_sasa']) * 100
    
    # Classify based on the 20% / 25% rule
    if relative_sasa < 20:
        location = "Buried / Core"
    elif relative_sasa > 25:
        location = "Surface / Exposed"
    else:
        location = "Intermediate"

**Calculate Distance to DNA and Display Results**

In [8]:
res_atoms = list(res.get_atoms())
distances = [np.linalg.norm(a1.get_coord() - a2.get_coord()) for a1 in res_atoms for a2 in dna_atoms]
min_dist = min(distances)
    
    # Print formatted output
print(f"Position {res_seq} ({mut['aa']}):")
print(f"  - Relative SASA: {relative_sasa:.1f}% -> {location}")
print(f"  - Distance to DNA: {min_dist:.1f} Å")
print("-" * 35)

Position 273 (ARG):
  - Relative SASA: 14.1% -> Buried / Core
  - Distance to DNA: 2.5 Å
-----------------------------------
